# Data Engineering: Import, Profiling and Cleaning


In [1]:
#importing libraries& modules  and defining paths
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT=Path('C:/Users/kipch/PycharmProjects/Customer_retention')
SRC_PATH=PROJECT_ROOT/"my_src"
sys.path.append(str(SRC_PATH))
from data_profiler import DataProfiler
from data_cleaner import DataCleaner

### Loading the source datasets


In [2]:
#loading datasets
raw_folder=PROJECT_ROOT/"datasets"/"raw_datasets"
accounts=pd.read_csv(raw_folder/'raw_accounts.csv')
churn_events=pd.read_csv(raw_folder/'raw_churn_events.csv')
feature_usage=pd.read_csv(raw_folder/'raw_feature_usage.csv')
subscriptions=pd.read_csv(raw_folder/'raw_subscriptions.csv')
support_tickets=pd.read_csv(raw_folder/'raw_support_tickets.csv')


In [3]:
#profiler
profiler1=DataProfiler(accounts,name="raw_accounts")
profiler2=DataProfiler(churn_events,name="raw_churn_events")
profiler3=DataProfiler(feature_usage,name="raw_feature_usage")
profiler4=DataProfiler(subscriptions,name="raw_subscriptions")
profiler5=DataProfiler(support_tickets,name="raw_support_tickets")

#cleaner
cleaner1=DataCleaner(accounts,name="raw_accounts")
cleaner2=DataCleaner(churn_events,name="raw_churn_events")
cleaner3=DataCleaner(feature_usage,name="raw_feature_usage")
cleaner4=DataCleaner(subscriptions,name="raw_subscriptions")
cleaner5=DataCleaner(support_tickets,name="raw_support_tickets")





## 1. Accounts: profile and clean


In [4]:
#head of the dataset
profiler1.quick_peek()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [5]:
#shape_summary
profiler1.shape_summary()

,dataset,rows,columns
0,raw_accounts,500,10


In [6]:
profiler1.column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,account_id,object,0,0.0,500,100.0
1,account_name,object,0,0.0,500,100.0
2,industry,object,0,0.0,5,1.0
3,country,object,0,0.0,7,1.4
4,signup_date,object,0,0.0,370,74.0
5,referral_source,object,0,0.0,5,1.0
6,plan_tier,object,0,0.0,3,0.6
7,seats,int64,0,0.0,73,14.6
8,is_trial,bool,0,0.0,2,0.4
9,churn_flag,bool,0,0.0,2,0.4


In [7]:
profiler1.duplicate_summary()

,dataset,duplicate_rows,duplicate_percent
0,raw_accounts,0,0.0


In [8]:
profiler1.numeric_summary()

,count,mean,std,min,25%,50%,75%,max
seats,500.0,20.56,21.044718,1.0,5.0,15.0,28.0,163.0


In [9]:
profiler1.outlier_summary_iqr()

,column,lower_bound,upper_bound,outlier_count,outlier_percent
0,seats,-29.5,62.5,25,5.0


In [10]:
profiler1.quick_peek()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [11]:
cat_cols=["account_name","industry","country","referral_source","plan_tier","is_trial","churn_flag"]
profiler1.cardinality_summary(cat_cols)

(            column  unique_count  unique_percent
 0     account_name           500           100.0
 1         industry             5             1.0
 2          country             7             1.4
 3  referral_source             5             1.0
 4        plan_tier             3             0.6
 5         is_trial             2             0.4
 6       churn_flag             2             0.4,
 {'industry':         industry  count  percentage
  0       DevTools    113        22.6
  1        FinTech    112        22.4
  2  Cybersecurity    100        20.0
  3     HealthTech     96        19.2
  4         EdTech     79        15.8,
  'country':   country  count  percentage
  0      US    291        58.2
  1      UK     58        11.6
  2      IN     49         9.8
  3      AU     32         6.4
  4      DE     25         5.0
  5      CA     23         4.6
  6      FR     22         4.4,
  'referral_source':   referral_source  count  percentage
  0         organic    114        22.8
 

In [12]:
cat_cols=["churn_flag","industry","country","referral_source","plan_tier","is_trial"]
profiler1.imbalance_summary(cat_cols)

,column,majority_class,majority_percent,minority_class,minority_percent,imbalance_ratio
0,churn_flag,False,78.0,True,22.0,3.55
1,industry,DevTools,22.6,EdTech,15.8,1.43
2,country,US,58.2,FR,4.4,13.23
3,referral_source,organic,22.8,partner,17.8,1.28
4,plan_tier,Pro,35.6,Enterprise,30.8,1.16
5,is_trial,False,80.6,True,19.4,4.15


In [13]:
accounts.columns

Index(['account_id', 'account_name', 'industry', 'country', 'signup_date',
       'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag'],
      dtype='object')

### Apply type and text standardization


In [14]:
#changing columns into their correct dtypes
text_cols=["account_name","industry","referral_source","plan_tier"]
type_mapping={
    "industry": "string",
    "country":"string",
    "signup_date":"datetime",
    "referral_source":"category",
    "plan_tier":"category",
    "seats":"integer",
}


In [15]:
#cleaning accounts_df
clean_folder = (
    PROJECT_ROOT
    / "datasets"
    / "cleaned_datasets"
)

(
    cleaner1
    .standardize_column_names()
    .convert_column_types(type_mapping)
    .strip_text_columns(text_cols)
    .lowercase_text_columns(text_cols)
    .save_clean_data(clean_folder)
)

Saved clean data -> C:\Users\kipch\PycharmProjects\Customer_retention\datasets\cleaned_datasets\raw_accounts_clean.csv


WindowsPath('C:/Users/kipch/PycharmProjects/Customer_retention/datasets/cleaned_datasets/raw_accounts_clean.csv')

In [16]:
cleaned_df1=cleaner1.get_clean_data()

In [17]:
cleaned_df1.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,company_0,edtech,US,2024-10-16,partner,basic,9,False,False
1,A-43a9e3,company_1,fintech,IN,2023-08-17,other,basic,18,False,True
2,A-0a282f,company_2,devtools,US,2024-08-27,organic,basic,1,False,False
3,A-1f0ac7,company_3,healthtech,UK,2023-08-27,other,basic,24,True,False
4,A-ce550d,company_4,healthtech,US,2024-10-27,event,enterprise,35,False,True


In [18]:
#cheking cleaned dataset
DataProfiler(cleaned_df1).column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,account_id,object,0,0.0,500,100.0
1,account_name,string,0,0.0,500,100.0
2,industry,string,0,0.0,5,1.0
3,country,string,0,0.0,7,1.4
4,signup_date,datetime64[ns],0,0.0,370,74.0
5,referral_source,string,0,0.0,5,1.0
6,plan_tier,string,0,0.0,3,0.6
7,seats,Int64,0,0.0,73,14.6
8,is_trial,bool,0,0.0,2,0.4
9,churn_flag,bool,0,0.0,2,0.4


In [19]:
cleaner1.get_cleaning_log()

,step,details
0,standardize_column_names,"{'old_columns': ['account_id', 'account_name',..."
1,convert_column_types,"{'industry': {'from': 'object', 'to': 'string'..."
2,strip_text_columns,"{'columns': ['account_name', 'industry', 'refe..."
3,lowercase_text_columns,"{'columns': ['account_name', 'industry', 'refe..."


## 2. Churn events: profile and clean


In [20]:
#quick glance
profiler2.quick_peek()


,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [21]:
#checking the shape
profiler2.shape_summary()


,dataset,rows,columns
0,raw_churn_events,600,9


In [22]:
#column_summary
profiler2.column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,churn_event_id,object,0,0.00,600,100.00
1,account_id,object,0,0.00,352,58.67
2,churn_date,object,0,0.00,311,51.83
3,reason_code,object,0,0.00,6,1.00
4,refund_amount_usd,float64,0,0.00,142,23.67
5,preceding_upgrade_flag,bool,0,0.00,2,0.33
6,preceding_downgrade_flag,bool,0,0.00,2,0.33
7,is_reactivation,bool,0,0.00,2,0.33
8,feedback_text,object,148,24.67,3,0.50


In [23]:
#checking for duplicates
profiler2.duplicate_summary(["churn_event_id"])

,dataset,duplicate_rows,duplicate_percent
0,raw_churn_events,0,0.0


In [24]:
#numeric summary
profiler2.numeric_summary()

,count,mean,std,min,25%,50%,75%,max
refund_amount_usd,600.0,14.420417,39.224591,0.0,0.0,0.0,0.0,392.92


In [25]:
#outlier_summary
profiler2.outlier_summary_iqr()

,column,lower_bound,upper_bound,outlier_count,outlier_percent
0,refund_amount_usd,0.0,0.0,142,23.67


In [26]:
#cardianlity summary
card_cols=('reason_code','preceding_upgrade_flag','preceding_downgrade_flag', 'is_reactivation', 'feedback_text')
profiler2.cardinality_summary(card_cols)


(                     column  unique_count  unique_percent
 0               reason_code             6            1.00
 1    preceding_upgrade_flag             2            0.33
 2  preceding_downgrade_flag             2            0.33
 3           is_reactivation             2            0.33
 4             feedback_text             3            0.50,
 {'reason_code':   reason_code  count  percentage
  0    features    114       19.00
  1     support    104       17.33
  2      budget    104       17.33
  3     unknown     95       15.83
  4  competitor     92       15.33
  5     pricing     91       15.17,
  'preceding_upgrade_flag':    preceding_upgrade_flag  count  percentage
  0                   False    477        79.5
  1                    True    123        20.5,
  'preceding_downgrade_flag':    preceding_downgrade_flag  count  percentage
  0                     False    547       91.17
  1                      True     53        8.83,
  'is_reactivation':    is_reactivation 

In [27]:
type_mapping2 = {

    "churn_event_id": "string",

    "account_id": "string",

    "churn_date": "datetime",

    "reason_code": "category",

    "refund_amount_usd": "float",

    "feedback_text": "string"
}

str_cols=['churn_event_id', 'account_id','reason_code','feedback_text']
lowc_cols=['reason_code','feedback_text']

In [28]:
#cleaning churn_events_df
clean_folder = (
    PROJECT_ROOT
    / "datasets"
    / "cleaned_datasets")
(
    cleaner2
    .standardize_column_names()
    .strip_text_columns(str_cols)
    .lowercase_text_columns(lowc_cols)
    .fill_missing_values(
        ["feedback_text"],
        method="constant",
        value="no_feedback"
    )
    .convert_column_types(type_mapping2)
    .save_clean_data(clean_folder)
)





Saved clean data -> C:\Users\kipch\PycharmProjects\Customer_retention\datasets\cleaned_datasets\raw_churn_events_clean.csv


WindowsPath('C:/Users/kipch/PycharmProjects/Customer_retention/datasets/cleaned_datasets/raw_churn_events_clean.csv')

In [29]:
df2=cleaner2.get_clean_data()

In [30]:
#checking data after cleaning
df2.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,no_feedback
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [31]:
#checking data after cleaning
DataProfiler(df2).column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,churn_event_id,string,0,0.0,600,100.00
1,account_id,string,0,0.0,352,58.67
2,churn_date,datetime64[ns],0,0.0,311,51.83
3,reason_code,category,0,0.0,6,1.00
4,refund_amount_usd,float64,0,0.0,142,23.67
5,preceding_upgrade_flag,bool,0,0.0,2,0.33
6,preceding_downgrade_flag,bool,0,0.0,2,0.33
7,is_reactivation,bool,0,0.0,2,0.33
8,feedback_text,string,0,0.0,4,0.67


In [32]:
#checking cleaning log
cleaner2.get_cleaning_log()

,step,details
0,standardize_column_names,"{'old_columns': ['churn_event_id', 'account_id..."
1,strip_text_columns,"{'columns': ['churn_event_id', 'account_id', '..."
2,lowercase_text_columns,"{'columns': ['reason_code', 'feedback_text']}"
3,fill_missing_values,"{'columns': ['feedback_text'], 'method': 'cons..."
4,convert_column_types,"{'churn_event_id': {'from': 'string', 'to': 's..."


## 3. Feature usage: profile and clean


In [33]:
profiler3.shape_summary()

,dataset,rows,columns
0,raw_feature_usage,25000,8


In [34]:
#head of df
profiler3.quick_peek()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


In [35]:
#column summary
profiler3.column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,usage_id,object,0,0.0,24979,99.92
1,subscription_id,object,0,0.0,4967,19.87
2,usage_date,object,0,0.0,731,2.92
3,feature_name,object,0,0.0,40,0.16
4,usage_count,int64,0,0.0,27,0.11
5,usage_duration_secs,int64,0,0.0,4168,16.67
6,error_count,int64,0,0.0,9,0.04
7,is_beta_feature,bool,0,0.0,2,0.01


In [36]:
#checking for duplicates based  on usage_id
profiler3.duplicate_summary(subset=["usage_id"])

,dataset,duplicate_rows,duplicate_percent
0,raw_feature_usage,0,0.0


In [37]:
#numeric_summary
profiler3.numeric_summary()

,count,mean,std,min,25%,50%,75%,max
usage_count,25000.0,10.02100,3.143729,0.0,8.0,10.0,12.0,26.0
usage_duration_secs,25000.0,3042.20288,2056.544615,0.0,1350.0,2760.0,4400.0,12696.0
error_count,25000.0,0.56428,1.012595,0.0,0.0,0.0,1.0,8.0


In [38]:
#outlier report
profiler3.outlier_summary_iqr()

,column,lower_bound,upper_bound,outlier_count,outlier_percent
0,usage_count,2.0,18.0,203,0.81
1,usage_duration_secs,-3225.0,8975.0,145,0.58
2,error_count,-1.5,2.5,1660,6.64


In [39]:
#checking for categorical/cardinality report
card_cols3=["feature_name","is_beta_feature"]
profiler3.cardinality_summary(card_cols3)

(            column  unique_count  unique_percent
 0     feature_name            40            0.16
 1  is_beta_feature             2            0.01,
 {'is_beta_feature':    is_beta_feature  count  percentage
  0            False  22456       89.82
  1             True   2544       10.18})

In [40]:
cols_strip3=["usage_id","subscription_id","feature_name"]
cols_lower3=["feature_name"]
type_mapping3={
    "usage_id":"string",
    "subscription_id":"string",
    "usage_date":"datetime",
    "feature_name":"string"
}

In [41]:
#cleaning feature_usage_df
clean_data_folder=PROJECT_ROOT/'datasets'/'cleaned_datasets'
(
    cleaner3.
    strip_text_columns(cols_strip3).
    lowercase_text_columns(cols_lower3).
    convert_column_types(type_mapping3).
    standardize_column_names().
    save_clean_data(clean_data_folder)
)

Saved clean data -> C:\Users\kipch\PycharmProjects\Customer_retention\datasets\cleaned_datasets\raw_feature_usage_clean.csv


WindowsPath('C:/Users/kipch/PycharmProjects/Customer_retention/datasets/cleaned_datasets/raw_feature_usage_clean.csv')

In [42]:
#checking the cleaned data
df3=cleaner3.get_clean_data()
df3.head(5)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


In [43]:
#checking column_summary after cleaning
DataProfiler(df3).column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,usage_id,string,0,0.0,24979,99.92
1,subscription_id,string,0,0.0,4967,19.87
2,usage_date,datetime64[ns],0,0.0,731,2.92
3,feature_name,string,0,0.0,40,0.16
4,usage_count,int64,0,0.0,27,0.11
5,usage_duration_secs,int64,0,0.0,4168,16.67
6,error_count,int64,0,0.0,9,0.04
7,is_beta_feature,bool,0,0.0,2,0.01


In [44]:
#clening log
cleaner3.get_cleaning_log()

,step,details
0,strip_text_columns,"{'columns': ['usage_id', 'subscription_id', 'f..."
1,lowercase_text_columns,{'columns': ['feature_name']}
2,convert_column_types,"{'usage_id': {'from': 'string', 'to': 'string'..."
3,standardize_column_names,"{'old_columns': ['usage_id', 'subscription_id'..."


## 4. Subscriptions: profile and clean


In [45]:
#shape
profiler4.shape_summary()

,dataset,rows,columns
0,raw_subscriptions,5000,14


In [46]:
#head of df
profiler4.quick_peek()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786.0,33432.0,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833.0,9996.0,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0.0,0.0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995.0,11940.0,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373.0,64476.0,False,False,False,False,monthly,True


In [47]:
profiler4.column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,subscription_id,object,0,0.00,5000,100.00
1,account_id,object,0,0.00,500,10.00
2,start_date,object,0,0.00,625,12.50
3,end_date,object,4514,90.28,208,4.16
4,plan_tier,object,0,0.00,3,0.06
5,seats,int64,0,0.00,123,2.46
6,mrr_amount,float64,0,0.00,306,6.12
7,arr_amount,float64,0,0.00,306,6.12
8,is_trial,bool,0,0.00,2,0.04
9,upgrade_flag,bool,0,0.00,2,0.04


In [48]:
profiler4.duplicate_summary(subset=["subscription_id"])

,dataset,duplicate_rows,duplicate_percent
0,raw_subscriptions,0,0.0


In [49]:
profiler4.numeric_summary()

,count,mean,std,min,25%,50%,75%,max
seats,5000.0,29.8520,23.089771,1.0,14.0,24.0,40.0,189.0
mrr_amount,5000.0,2267.7494,3421.375348,0.0,285.0,931.0,2786.0,33830.0
arr_amount,5000.0,27212.9928,41056.504178,0.0,3420.0,11172.0,33432.0,405960.0


In [50]:
profiler4.outlier_summary_iqr()

,column,lower_bound,upper_bound,outlier_count,outlier_percent
0,seats,-25.0,79.0,216,4.32
1,mrr_amount,-3466.5,6537.5,471,9.42
2,arr_amount,-41598.0,78450.0,471,9.42


In [51]:
card_cols4=["plan_tier",'is_trial', 'upgrade_flag','downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']
profiler4.cardinality_summary(card_cols4)

(              column  unique_count  unique_percent
 0          plan_tier             3            0.06
 1           is_trial             2            0.04
 2       upgrade_flag             2            0.04
 3     downgrade_flag             2            0.04
 4         churn_flag             2            0.04
 5  billing_frequency             2            0.04
 6    auto_renew_flag             2            0.04,
 {'plan_tier':     plan_tier  count  percentage
  0  Enterprise   1723       34.46
  1         Pro   1675       33.50
  2       Basic   1602       32.04,
  'is_trial':    is_trial  count  percentage
  0     False   4222       84.44
  1      True    778       15.56,
  'upgrade_flag':    upgrade_flag  count  percentage
  0         False   4471       89.42
  1          True    529       10.58,
  'downgrade_flag':    downgrade_flag  count  percentage
  0           False   4782       95.64
  1            True    218        4.36,
  'churn_flag':    churn_flag  count  percentage
  0 

In [52]:
cols_strip4=[ "subscription_id","account_id","plan_tier","billing_frequency"]
cols_lower4=["plan_tier","billing_frequency"]
type_mapping4 = {

    "subscription_id": "string",
    "account_id": "string",
    "start_date": "datetime",
    "end_date": "datetime",
    "plan_tier": "category",
    "billing_frequency": "category",
}


In [53]:
#cleaning  df 4
clean_data_folder=PROJECT_ROOT/"datasets"/"cleaned_datasets"
(
    cleaner4.
    convert_column_types(type_mapping4).
    standardize_column_names().
    strip_text_columns(cols_strip4).
    lowercase_text_columns(cols_lower4).
    save_clean_data(clean_data_folder)
)


Saved clean data -> C:\Users\kipch\PycharmProjects\Customer_retention\datasets\cleaned_datasets\raw_subscriptions_clean.csv


WindowsPath('C:/Users/kipch/PycharmProjects/Customer_retention/datasets/cleaned_datasets/raw_subscriptions_clean.csv')

In [54]:
#checking cleaning data
df4=cleaner4.get_clean_data()
df4.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,enterprise,14,2786.0,33432.0,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaT,pro,17,833.0,9996.0,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaT,enterprise,62,0.0,0.0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,enterprise,5,995.0,11940.0,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaT,enterprise,27,5373.0,64476.0,False,False,False,False,monthly,True


In [55]:
#column summary of cleaned data
DataProfiler(df4).column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,subscription_id,string,0,0.00,5000,100.00
1,account_id,string,0,0.00,500,10.00
2,start_date,datetime64[ns],0,0.00,625,12.50
3,end_date,datetime64[ns],4514,90.28,208,4.16
4,plan_tier,string,0,0.00,3,0.06
5,seats,int64,0,0.00,123,2.46
6,mrr_amount,float64,0,0.00,306,6.12
7,arr_amount,float64,0,0.00,306,6.12
8,is_trial,bool,0,0.00,2,0.04
9,upgrade_flag,bool,0,0.00,2,0.04


In [56]:
#cleaning_log
cleaner4.get_cleaning_log()

,step,details
0,convert_column_types,"{'subscription_id': {'from': 'object', 'to': '..."
1,standardize_column_names,"{'old_columns': ['subscription_id', 'account_i..."
2,strip_text_columns,"{'columns': ['subscription_id', 'account_id', ..."
3,lowercase_text_columns,"{'columns': ['plan_tier', 'billing_frequency']}"


## 5. Support tickets: profile and clean


In [57]:
profiler5.shape_summary()

,dataset,rows,columns
0,raw_support_tickets,2000,9


In [58]:
profiler5.quick_peek()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False


In [59]:
profiler5.column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,ticket_id,object,0,0.00,2000,100.00
1,account_id,object,0,0.00,492,24.60
2,submitted_at,object,0,0.00,671,33.55
3,closed_at,object,0,0.00,1893,94.65
4,resolution_time_hours,float64,0,0.00,72,3.60
5,priority,object,0,0.00,4,0.20
6,first_response_time_minutes,int64,0,0.00,180,9.00
7,satisfaction_score,float64,825,41.25,3,0.15
8,escalation_flag,bool,0,0.00,2,0.10


In [60]:
profiler5.duplicate_summary(subset=["ticket_id"])

,dataset,duplicate_rows,duplicate_percent
0,raw_support_tickets,0,0.0


In [61]:
profiler5.numeric_summary()

,count,mean,std,min,25%,50%,75%,max
resolution_time_hours,2000.0,35.861000,21.138427,1.0,17.0,35.0,54.0,72.0
first_response_time_minutes,2000.0,88.480000,51.531877,1.0,43.0,88.0,131.0,180.0
satisfaction_score,1175.0,3.981277,0.809646,3.0,3.0,4.0,5.0,5.0


In [62]:
profiler5.outlier_summary_iqr()

,column,lower_bound,upper_bound,outlier_count,outlier_percent
0,resolution_time_hours,-38.5,109.5,0,0.0
1,first_response_time_minutes,-89.0,263.0,0,0.0
2,satisfaction_score,0.0,8.0,0,0.0


In [63]:
card_cols5=["priority","escalation_flag"]
cols_strip5=["ticket_id","account_id","priority"]
cols_lower5=["priority"]
type_mapping5={
    "ticket_id":"string",
    "account_id":"string",
    "submitted_at":"datetime",
    "closed_at":"datetime",
    "priority":"category",
}

In [64]:
profiler5.cardinality_summary(card_cols5)

(            column  unique_count  unique_percent
 0         priority             4             0.2
 1  escalation_flag             2             0.1,
 {'priority':   priority  count  percentage
  0   urgent    514       25.70
  1     high    510       25.50
  2   medium    491       24.55
  3      low    485       24.25,
  'escalation_flag':    escalation_flag  count  percentage
  0            False   1905       95.25
  1             True     95        4.75})

In [65]:
CLEAN_FOLDER= PROJECT_ROOT/'datasets'/'cleaned_datasets'

(
    cleaner5.
    convert_column_types(type_mapping5).
    strip_text_columns(cols_strip5).
    lowercase_text_columns(cols_lower5).
    standardize_column_names().
    save_clean_data(CLEAN_FOLDER)
)

Saved clean data -> C:\Users\kipch\PycharmProjects\Customer_retention\datasets\cleaned_datasets\raw_support_tickets_clean.csv


WindowsPath('C:/Users/kipch/PycharmProjects/Customer_retention/datasets/cleaned_datasets/raw_support_tickets_clean.csv')

In [66]:
#check df head
df5=cleaner5.get_clean_data()
df5.head(5)

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False


In [67]:
#checking column summary
DataProfiler(df5).column_summary()

,column,dtype,missing_count,missing_percent,unique_count,unique_percent
0,ticket_id,string,0,0.00,2000,100.00
1,account_id,string,0,0.00,492,24.60
2,submitted_at,datetime64[ns],0,0.00,671,33.55
3,closed_at,datetime64[ns],0,0.00,1893,94.65
4,resolution_time_hours,float64,0,0.00,72,3.60
5,priority,string,0,0.00,4,0.20
6,first_response_time_minutes,int64,0,0.00,180,9.00
7,satisfaction_score,float64,825,41.25,3,0.15
8,escalation_flag,bool,0,0.00,2,0.10


In [68]:
#getting the cleaning log
cleaner5.get_cleaning_log()

,step,details
0,convert_column_types,"{'ticket_id': {'from': 'object', 'to': 'string..."
1,strip_text_columns,"{'columns': ['ticket_id', 'account_id', 'prior..."
2,lowercase_text_columns,{'columns': ['priority']}
3,standardize_column_names,"{'old_columns': ['ticket_id', 'account_id', 's..."


## 6. Save and load cleaned data


In [69]:
#importing  required modules
from db_connector import (
    DatabaseConnector,
    MySQLTableExtractor,
    MySQLTableLoader
)

In [70]:
clean_connector = DatabaseConnector(
    database="cleaned_ravenstack_dw"
)

clean_engine = clean_connector.get_engine()

In [71]:
loader = MySQLTableLoader(
    clean_engine
)
clean_folder = (
    PROJECT_ROOT
    / "datasets"
    / "cleaned_datasets"
)

loader.load_all_cleaned_csvs(
    clean_folder,
    if_exists="replace"
)

Loaded 500 rows into raw_accounts
Loaded 600 rows into raw_churn_events
Loaded 25,000 rows into raw_feature_usage
Loaded 5,000 rows into raw_subscriptions
Loaded 2,000 rows into raw_support_tickets


### Verify saved-file consistency


In [72]:
# 1. Data directly inside cleaner
cleaned_df1.dtypes

account_id                 object
account_name       string[python]
industry           string[python]
country            string[python]
signup_date        datetime64[ns]
referral_source    string[python]
plan_tier          string[python]
seats                       Int64
is_trial                     bool
churn_flag                   bool
dtype: object

In [73]:
# 2. Same data after reading the saved cleaned CSV
clean_df_path=PROJECT_ROOT/"datasets"/"cleaned_datasets"
pd.read_csv(clean_df_path/"raw_accounts_clean.csv").dtypes

account_id         object
account_name       object
industry           object
country            object
signup_date        object
referral_source    object
plan_tier          object
seats               int64
is_trial             bool
churn_flag           bool
dtype: object